# Medical Rhythms (Synthetic ECG) — QIH Period Features

**Purpose:** Illustrate how QIH (analytic QFT sampling of periodic state) can provide
compact, explainable features for detecting rhythm irregularities.

**What we do:**
1. Generate synthetic ECG-like waveform with heart-rate variability (HRV).
2. Inject occasional ectopic beats / irregular intervals.
3. Compare classical FFT to **QIH histograms** derived from your `PeriodicState` at the estimated period.
4. Train a tiny classifier (if scikit-learn is present) to flag irregular windows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_hybrid_system.tools_qih.qih_pat import qih_pat_sequence_features

rng = np.random.default_rng(42)

# --- ECG-like synthesis ---
def synth_ecg(T=4000, base_period=60, hrv_std=3, ectopic_prob=0.02, noise=0.05):
    # generate impulses at variable intervals -> convolve with a PQRST-ish kernel
    t = 0
    impulses = np.zeros(T)
    while t < T:
        impulses[int(t)] = 1.0
        # RR interval with variability
        rr = base_period + rng.normal(0, hrv_std)
        # occasional ectopic short interval
        if rng.random() < ectopic_prob:
            rr *= 0.6
        t += max(5, int(rr))
    # PQRST kernel (very rough)
    k = np.array([0.1, 0.2, 0.5, 1.0, 0.5, 0.2, 0.1])
    ecg = np.convolve(impulses, k, mode="same")
    ecg += noise * rng.normal(size=T)
    return ecg

T = 8000
clean = synth_ecg(T=T, base_period=70, hrv_std=2, ectopic_prob=0.0)
irreg = synth_ecg(T=T, base_period=70, hrv_std=6, ectopic_prob=0.08)

plt.figure()
plt.plot(clean[:800], label="clean")
plt.plot(irreg[:800], label="irregular", alpha=0.8)
plt.title("Synthetic ECG-like signals (first 800 samples)")
plt.legend(); plt.xlabel("time"); plt.ylabel("amplitude"); plt.show()

# --- Windowing ---
W, S = 256, 128
def windows(x):
    for start in range(0, len(x)-W+1, S):
        yield x[start:start+W]

X0 = np.vstack(list(windows(clean)))
X1 = np.vstack(list(windows(irreg)))
y0 = np.zeros(len(X0), dtype=int)
y1 = np.ones(len(X1), dtype=int)

X = np.vstack([X0, X1])
y = np.concatenate([y0, y1])
perm = rng.permutation(len(X)); X, y = X[perm], y[perm]

# --- Baseline FFT feature ---
def fft_topbin(Wx):
    import numpy as np
    N = Wx.shape[1]
    xf = np.fft.rfftfreq(N, d=1.0)
    mags = np.abs(np.fft.rfft(Wx - Wx.mean(axis=1, keepdims=True), axis=1))
    mags[:,0] = 0.0
    idx = np.argmax(mags, axis=1)
    return xf[idx]

fft_feat = fft_topbin(X).reshape(-1,1)

# --- QIH features ---
H, periods = qih_pat_sequence_features(X, win=W, stride=W, n=10, shots=1024, bins=64)

print("Shapes:", X.shape, H.shape, fft_feat.shape)

# --- Classifier ---
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    split = int(0.8*len(X))
    Htr, Hte = H[:split], H[split:]
    Ftr, Fte = fft_feat[:split], fft_feat[split:]
    ytr, yte = y[:split], y[split:]
    base = LogisticRegression(max_iter=300).fit(Ftr, ytr)
    qih  = LogisticRegression(max_iter=300).fit(Htr, ytr)
    a_base = roc_auc_score(yte, base.predict_proba(Fte)[:,1])
    a_qih  = roc_auc_score(yte, qih.predict_proba(Hte)[:,1])
    print({"auc_fft_topbin": float(a_base), "auc_qih_hist": float(a_qih)})
except Exception as e:
    print("sklearn not available:", e)

# --- Visualization of class means ---
import matplotlib.pyplot as plt
plt.figure()
plt.plot(H[y==0].mean(axis=0), label="clean")
plt.plot(H[y==1].mean(axis=0), label="irregular")
plt.title("Mean QIH histogram by class")
plt.xlabel("bin"); plt.ylabel("probability"); plt.legend(); plt.show()